In [1]:
import os
import re
import json
import uuid
import hashlib
import time
from typing import Literal
from collections import Counter
from dotenv import load_dotenv
from pydantic import BaseModel
from tqdm import tqdm

from groq import Groq
import cohere
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocItemLabel

load_dotenv()

# Clients
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
cohere_client = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

print("✓ Groq client ready")
print("✓ Cohere client ready")
print("✓ Environment loaded")

c:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Groq client ready
✓ Cohere client ready
✓ Environment loaded


Step 2 : Filing registry
 

In [2]:
FILINGS = [
    {
        "company": "Apple Inc.",
        "ticker": "AAPL",
        "exchange": "NASDAQ",
        "filing_type": "10-Q",
        "period": "Q2-FY2026",
        "filing_date": "2026-03-28",
        "currency": "USD",
        "accounting_standard": "US GAAP",
        "fiscal_year_end": "September",
        "local_path": "../data/raw/aapl_10q_q2_2026.pdf"
    },
    {
        "company": "Microsoft Corporation",
        "ticker": "MSFT",
        "exchange": "NASDAQ",
        "filing_type": "10-Q",
        "period": "Q1-FY2026",
        "filing_date": "2025-12-31",
        "currency": "USD",
        "accounting_standard": "US GAAP",
        "fiscal_year_end": "June",
        "local_path": "../data/raw/msft_10q_q1_2026.pdf"
    }
]

print(f"Filings registered: {len(FILINGS)}")
for f in FILINGS:
    path_exists = os.path.exists(f["local_path"])
    print(f"  {f['ticker']} | {f['period']} | {f['filing_date']} | file: {path_exists}")

Filings registered: 2
  AAPL | Q2-FY2026 | 2026-03-28 | file: True
  MSFT | Q1-FY2026 | 2025-12-31 | file: True


Step 3 :  Docling conversion
 

In [3]:
converter = DocumentConverter()
all_results = {}

for filing in FILINGS:
    print(f"Converting {filing['company']}...")
    result = converter.convert(filing["local_path"])
    all_results[filing["ticker"]] = {
        "result": result,
        "metadata": {
            "company": filing["company"],
            "ticker": filing["ticker"],
            "period": filing["period"],
            "filing_date": filing["filing_date"],
            "filing_type": filing["filing_type"],
            "currency": filing["currency"],
            "source": filing["local_path"].split("/")[-1]
        }
    }
    print(f"  ✓ {filing['ticker']} converted")

print("\nAll documents converted.")

Converting Apple Inc....


[INFO] 2026-07-20 12:56:07,314 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 12:56:07,362 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-20 12:56:07,422 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 12:56:07,423 [RapidOCR] main.py:50: Using C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 12:56:07,777 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 12:56:07,779 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-20 12:56:07,785 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-20 12:

  ✓ AAPL converted
Converting Microsoft Corporation...
  ✓ MSFT converted

All documents converted.


Step 4 : Document cleaning

In [4]:
def clean_prose(text: str) -> str:
    """
    cleaning for SEC filing prose.
    Removes noise introduced by PDF extraction.
    """
    # Fix broken lines — sentence split across PDF lines
    text = re.sub(r'(?<=[a-z,;])\n(?=[a-z])', ' ', text)
    
    # Remove page number patterns (standalone numbers)
    text = re.sub(r'\n\d{1,3}\n', '\n', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    
    # Remove common repeated header/footer patterns
    text = re.sub(r'(Apple Inc\.|Microsoft Corporation)\s*\n', '', text)
    text = re.sub(r'Form 10-Q\s*\n', '', text)
    text = re.sub(r'Table of Contents\s*\n', '', text)
    
    return text.strip()


# Test once Docling finishes
print("✓ clean_prose() ready")


✓ clean_prose() ready


Step 6 — Section detection

In [5]:
class SectionClassification(BaseModel):
    section: Literal[
        "Income Statement", "Balance Sheet", "Cash Flow",
        "Shareholders Equity", "MD&A", "Risk Factors",
        "Legal Proceedings", "Notes", "Signature", "Exhibits", "General"
    ]
    confidence: Literal["high", "low"]


SECTION_PATTERNS = {
    "Income Statement": [
        "statements of operations", "statements of income",
        "income statements", "income statement", "comprehensive income"
    ],
    "Balance Sheet": [
        "balance sheet", "financial position", "balance sheets"
    ],
    "Cash Flow": [
        "cash flows", "cash flow statement", "cash flows statements"
    ],
    "Shareholders Equity": [
        "shareholders equity", "stockholders equity",
        "stockholders' equity statements", "equity statements"
    ],
    "MD&A": [
        "management's discussion", "results of operations",
        "summary results of operations", "item 2. management",
        "item 2.    management"
    ],
    "Risk Factors": [
        "risk factors", "item 1a", "item 1a.",
        "strategic and competitive risks"
    ],
    "Legal Proceedings": [
        "legal proceedings", "item 1. legal", "legal proceeding"
    ],
    "Notes": [
        "notes to condensed", "notes to consolidated",
        "notes to financial statements"
    ],
    "Signature": ["pursuant to the requirements", "duly authorized"],
    "Exhibits": ["exhibit 31", "exhibit 32", "incorporated by reference"]
}

In [6]:
CACHE_PATH = "../data/processed/section_cache.json"

def load_section_cache() -> dict:
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, "r") as f:
            print(f"✓ Section cache loaded from disk")
            return json.load(f)
    return {}

def save_section_cache():
    os.makedirs("../data/processed", exist_ok=True)
    with open(CACHE_PATH, "w") as f:
        json.dump(section_cache, f, indent=2)
    print(f"✓ Section cache saved: {len(section_cache)} entries")

# Load on startup
section_cache = load_section_cache()
print(f"  Cache entries: {len(section_cache)}")

def detect_section_regex(text: str) -> str:
    text_lower = text.lower()
    for section, patterns in SECTION_PATTERNS.items():
        for pattern in patterns:
            if pattern in text_lower:
                return section
    return "General"

✓ Section cache loaded from disk
  Cache entries: 219


In [7]:
system_prompt = """You are a financial document section classifier for SEC 10-Q filings.
Classify ONLY top-level major section headers. Sub-headers → General.

High confidence examples:
  "INCOME STATEMENTS" → Income Statement
  "BALANCE SHEETS" → Balance Sheet
  "STOCKHOLDERS' EQUITY STATEMENTS" → Shareholders Equity
  "NOTES TO FINANCIAL STATEMENTS" → Notes
  "CASH FLOWS STATEMENTS" → Cash Flow
  "MANAGEMENT'S DISCUSSION AND ANALYSIS" → MD&A
  "ITEM 1A. RISK FACTORS" → Risk Factors
  "ITEM 1. LEGAL PROCEEDINGS" → Legal Proceedings

Low confidence → General:
  Individual notes, sub-topics, navigation text, date headers

Return ONLY valid JSON: {"section": "...", "confidence": "high|low"}"""

In [8]:
def detect_section_llm(header_text: str) -> str:
    if header_text in section_cache:
        return section_cache[header_text]

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": f"Classify: '{header_text}'"
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    try:
        result = json.loads(response.choices[0].message.content)
        section = result.get("section", "General")
        confidence = result.get("confidence", "low")
        final = section if confidence == "high" else "General"
        section_cache[header_text] = final
        return final
    except:
        section_cache[header_text] = "General"
        return "General"
    
def detect_section(header_text: str) -> str:
    """Hybrid: regex fast path → Groq LLM fallback."""
    regex_result = detect_section_regex(header_text)
    if regex_result != "General":
        return regex_result
    return detect_section_llm(header_text)


# Test
test_headers = [
    "INCOME STATEMENTS",
    "CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS",
    "BALANCE SHEETS",
    "NOTES TO FINANCIAL STATEMENTS",
    "MANAGEMENT'S DISCUSSION AND ANALYSIS",
    "ITEM 1A. RISK FACTORS",
    "PART I Item 1",
    "Note 2 - Revenue"
]

print("Testing section detector:\n")
for h in test_headers:
    result = detect_section(h)
    print(f"  [{result:25}] '{h}'")

Testing section detector:

  [Income Statement         ] 'INCOME STATEMENTS'
  [Income Statement         ] 'CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS'
  [Balance Sheet            ] 'BALANCE SHEETS'
  [Notes                    ] 'NOTES TO FINANCIAL STATEMENTS'
  [MD&A                     ] 'MANAGEMENT'S DISCUSSION AND ANALYSIS'
  [Risk Factors             ] 'ITEM 1A. RISK FACTORS'
  [General                  ] 'PART I Item 1'
  [General                  ] 'Note 2 - Revenue'


Helper Functions

In [9]:
def get_page_no(element) -> int:
    try:
        return element.prov[0].page_no
    except:
        return None


def get_part_and_item(current_section: str, text: str) -> dict:
    """Extract Part and Item from section boundary text."""
    text_lower = text.lower()
    part = None
    item = None

    if "part i" in text_lower and "part ii" not in text_lower:
        part = "Part I"
    elif "part ii" in text_lower:
        part = "Part II"

    for i in range(1, 5):
        if f"item {i}." in text_lower or f"item {i} " in text_lower:
            item = f"Item {i}"
            break

    return {"part": part, "item": item}


def split_into_sentences(text: str) -> list:
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def is_meaningful_row(row: str) -> bool:
    cells = [c.strip() for c in row.split('|') if c.strip()]
    if not cells:
        return False
    non_empty = [c for c in cells if c and c != '-']
    if len(non_empty) < 2:
        return False
    date_patterns = [
        'march', 'june', 'september', 'december',
        'january', 'february', 'april', 'july',
        'august', 'october', 'november'
    ]
    if any(month in row.lower() for month in date_patterns):
        return False
    has_financial = any(
        '$' in cell or
        (any(char.isdigit() for char in cell) and ',' in cell)
        for cell in cells
    )
    return has_financial


def split_table_into_rows(table_markdown: str) -> list:
    lines = table_markdown.strip().split('\n')
    header = None
    separator = None
    data_rows = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if re.match(r'^\|[-|\s:]+\|$', line):
            separator = line
            continue
        if header is None:
            header = line
            continue
        if line.startswith('|'):
            data_rows.append(line)

    children = []
    for row in data_rows:
        if not is_meaningful_row(row):
            continue
        child_content = f"{header}\n{separator}\n{row}" if separator else f"{header}\n{row}"
        children.append(child_content)
    return children


def make_chunk_id(ticker: str, index: int) -> str:
    raw = f"{ticker}_{index}"
    return hashlib.md5(raw.encode()).hexdigest()[:16]


SECTION_BOUNDARY_PHRASES = [
    "item 1a", "risk factor",
    "item 1. legal", "legal proceeding",
    "the company is subject to various legal",
    "strategic and competitive risks",
    "this item and other sections of this quarterly report",
    "part ii. other", "part ii other"
]



Step 7 : Parent-Child chunking

In [10]:
def create_parent_child_chunks(ticker: str, result, filing_metadata: dict):
    parents = {}
    children = []
    current_section = "General"
    current_part = None
    current_item = None
    doc = result.document
    child_index = 0

    for element, level in doc.iterate_items():
        page_no = get_page_no(element)

        # Skip page headers and footers
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.PAGE_HEADER,
            DocItemLabel.PAGE_FOOTER
        ]:
            continue

        # Section detection from SECTION_HEADER and TITLE
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.SECTION_HEADER,
            DocItemLabel.TITLE
        ]:
            if hasattr(element, 'text'):
                detected = detect_section(element.text)
                if detected != "General":
                    current_section = detected
                # Extract part and item
                pi = get_part_and_item(current_section, element.text)
                if pi["part"]:
                    current_part = pi["part"]
                if pi["item"]:
                    current_item = pi["item"]
            continue

        # Rich metadata for every chunk
        base_metadata = {
            **filing_metadata,
            "section": current_section,
            "part": current_part,
            "item": current_item,
            "page": page_no,
        }

        # TABLE — atomic parent + row children
        if hasattr(element, 'data'):
            table_text = element.export_to_markdown(doc=doc)
            if not table_text.strip():
                continue

            parent_id = str(uuid.uuid4())
            parents[parent_id] = {
                "content": table_text,
                "metadata": {**base_metadata, "chunk_type": "table"}
            }

            row_children = split_table_into_rows(table_text)
            for row in row_children:
                children.append({
                    "content": row,
                    "metadata": {
                        **base_metadata,
                        "chunk_type": "table",
                        "parent_id": parent_id,
                        "child_index": child_index
                    }
                })
                child_index += 1
            continue

        # PROSE — clean + parent + sentence pair children
        if hasattr(element, 'text') and element.text.strip():
            prose_text = clean_prose(element.text.strip())
            if not prose_text:
                continue

            text_lower = prose_text.lower()

            # Check for section boundary in prose text
            is_boundary = any(
                phrase in text_lower
                for phrase in SECTION_BOUNDARY_PHRASES
            )
            if is_boundary:
                detected = detect_section(prose_text[:200])
                if detected != "General":
                    current_section = detected
                    continue

            parent_id = str(uuid.uuid4())
            parents[parent_id] = {
                "content": prose_text,
                "metadata": {**base_metadata, "chunk_type": "prose"}
            }

            sentences = split_into_sentences(prose_text)
            for i in range(0, len(sentences), 2):
                pair = sentences[i:i + 2]
                child_text = " ".join(pair)
                if child_text.strip():
                    children.append({
                        "content": child_text,
                        "metadata": {
                            **base_metadata,
                            "chunk_type": "prose",
                            "parent_id": parent_id,
                            "child_index": child_index
                        }
                    })
                    child_index += 1

    return parents, children


print("✓ create_parent_child_chunks() ready")

✓ create_parent_child_chunks() ready


Step 8 : Run chunking for all filings

In [11]:
all_parents = {}
all_children = {}

for filing in FILINGS:
    ticker = filing["ticker"]
    result = all_results[ticker]["result"]
    filing_metadata = all_results[ticker]["metadata"]

    print(f"Chunking {filing['company']}...")
    parents, children = create_parent_child_chunks(
        ticker, result, filing_metadata
    )

    all_parents[ticker] = parents
    all_children[ticker] = children

    sections = Counter(c["metadata"]["section"] for c in children)
    print(f"  Parents : {len(parents)}")
    print(f"  Children: {len(children)}")
    print(f"  Sections:")
    for section, count in sections.most_common():
        print(f"    {section}: {count}")
    print()

# Save section cache after chunking
save_section_cache()

total_parents = sum(len(p) for p in all_parents.values())
total_children = sum(len(c) for c in all_children.values())
print(f"Total parents : {total_parents}")
print(f"Total children: {total_children}")

Chunking Apple Inc....
  Parents : 193
  Children: 437
  Sections:
    Notes: 97
    MD&A: 95
    Legal Proceedings: 48
    General: 39
    Risk Factors: 39
    Income Statement: 32
    Balance Sheet: 30
    Cash Flow: 28
    Shareholders Equity: 17
    Exhibits: 12

Chunking Microsoft Corporation...
  Parents : 463
  Children: 1002
  Sections:
    Risk Factors: 234
    MD&A: 203
    Shareholders Equity: 193
    Income Statement: 171
    Cash Flow: 76
    Notes: 45
    General: 38
    Balance Sheet: 34
    Legal Proceedings: 3
    Exhibits: 3
    Controls and Procedures: 2

✓ Section cache saved: 219 entries
Total parents : 656
Total children: 1439


In [13]:
os.makedirs("../data/processed", exist_ok=True)

docstore = {}
for ticker, parents in all_parents.items():
    docstore.update(parents)

with open("../data/processed/docstore.json", "w") as f:
    json.dump(docstore, f)

print(f"✓ Docstore saved: {len(docstore)} parents")
print(f"  AAPL: {len(all_parents['AAPL'])} parents")
print(f"  MSFT: {len(all_parents['MSFT'])} parents")

✓ Docstore saved: 656 parents
  AAPL: 193 parents
  MSFT: 463 parents


Step x — Contextual enrichment


Step x — Enrich all children

 Step 9 : Pinecone setup

In [24]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Delete old index and create new one with correct dimensions
INDEX_NAME = "financial-reconciliation-v4"
EMBEDDING_DIM = 384  # BAAI/bge-small-en-v1.5

existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating index: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("✓ Index created")
else:
    print(f"✓ Index exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)
print(f"✓ Connected to: {INDEX_NAME}")
print(index.describe_index_stats())

Creating index: financial-reconciliation-v4
✓ Index created
✓ Connected to: financial-reconciliation-v4
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


 Step 10 : Cohere embeddings + BM25 + upsert

In [25]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
EMBEDDING_DIM = 384
BATCH_SIZE = 96

def get_embeddings(texts: list, input_type: str = "search_document") -> list:
    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return embeddings.tolist()

def scale_sparse(vector: dict, alpha: float) -> dict:
    return {
        "indices": vector["indices"],
        "values": [v * (1 - alpha) for v in vector["values"]]
    }


def scale_dense(vector: list, alpha: float) -> list:
    return [v * alpha for v in vector]

def clean_metadata(metadata: dict) -> dict:
    """Remove None values from metadata — Pinecone rejects null."""
    return {k: v for k, v in metadata.items() if v is not None}


def upsert_children(ticker: str, children: list, alpha: float = 0.5):
    namespace = f"{ticker}_PC"
    print(f"Upserting {ticker} → namespace: {namespace}")
    print(f"Total children: {len(children)}")

    skipped = 0
    for i in tqdm(range(0, len(children), BATCH_SIZE)):
        batch = children[i:i + BATCH_SIZE]
        texts = [c["content"] for c in batch]

        dense_embeddings = get_embeddings(texts, input_type="search_document")
        sparse_embeddings = bm25_encoder.encode_documents(texts)

        vectors = []
        for child, dense, sparse in zip(batch, dense_embeddings, sparse_embeddings):
            if not sparse["values"]:
                skipped += 1
                continue

            chunk_id = make_chunk_id(ticker, child["metadata"]["child_index"])

            vectors.append({
                "id": chunk_id,
                "values": scale_dense(dense, alpha),
                "sparse_values": scale_sparse(sparse, alpha),
                "metadata": {
                    **clean_metadata(child["metadata"]),
                    "content": child["content"][:1000]
                }
            })

        if vectors:
            index.upsert(vectors=vectors, namespace=namespace)

    print(f"  ✓ Done | skipped: {skipped} empty sparse vectors")


# Fit BM25 on all children corpus

all_child_texts = []
for ticker, children in all_children.items():
    for child in children:
        all_child_texts.append(child["content"])

print(f"Fitting BM25 on {len(all_child_texts)} children...")
bm25_encoder = BM25Encoder()
bm25_encoder.fit(all_child_texts)
print("✓ BM25 fitted")

# Upsert all children
print("\nUpserting to Pinecone...")
for ticker, children in all_children.items():
    upsert_children(ticker, children)

print("\nFinal index stats:")
print(index.describe_index_stats())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5384.53it/s]


Fitting BM25 on 1439 children...


100%|██████████| 1439/1439 [00:00<00:00, 2154.04it/s]


✓ BM25 fitted

Upserting to Pinecone...
Upserting AAPL → namespace: AAPL_PC
Total children: 437


100%|██████████| 5/5 [00:32<00:00,  6.52s/it]


  ✓ Done | skipped: 2 empty sparse vectors
Upserting MSFT → namespace: MSFT_PC
Total children: 1002


100%|██████████| 11/11 [01:04<00:00,  5.88s/it]


  ✓ Done | skipped: 1 empty sparse vectors

Final index stats:
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL_PC': {'vector_count': 435},
                'MSFT_PC': {'vector_count': 1001}},
 'total_vector_count': 1436,
 'vector_type': 'dense'}


In [26]:
bm25_encoder.dump("../data/processed/bm25_encoder.json")
print("✓ BM25 encoder saved")

✓ BM25 encoder saved


Step 11 : Query Router

In [32]:
class QueryRoute(BaseModel):
    chunk_type: Literal["table", "prose"]
    section: Literal[
        "Income Statement", "Balance Sheet", "Cash Flow",
        "MD&A", "Risk Factors", "Legal Proceedings",
        "Notes", "Shareholders Equity", "Any"
    ]


def route_query(query: str) -> dict:
    system_prompt = """You are a financial document query router for SEC 10-Q filings.

PRIORITY RULES — apply these first:
- inventory/inventories → Balance Sheet, table
- dividends PAID as cash outflow → Cash Flow, table
- R&D as percentage or ratio → MD&A, prose
- EPS computation or weighted average shares → Notes, table
- gross margin as dollar figure → Income Statement, table
- R&D dollar amount → Income Statement, table
- change/grew/increased about a metric → MD&A, prose
- share repurchase program details, RSU activity → Notes, prose

SECTION GUIDE:
- Income Statement: revenue, net sales, gross margin, operating income, net income, cost of sales
- Balance Sheet: total assets, cash, liabilities, equity, receivables, inventory
- Cash Flow: operating activities, investing, financing, dividends paid, capex
- MD&A: why metrics changed, percentage change, growth reasons, segment performance
- Risk Factors: risks, uncertainties, challenges
- Legal Proceedings: lawsuits, investigations
- Notes: RSUs, debt details, EPS computation, weighted average shares
- Shareholders Equity: retained earnings, AOCI
- Any: spans multiple sections

CHUNK TYPE: NUMBER/FIGURE → table | EXPLANATION/WHY/HOW → prose

Return ONLY valid JSON: {"chunk_type": "table", "section": "Income Statement"}"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    try:
        result = json.loads(response.choices[0].message.content)
        # Validate against schema
        route = QueryRoute(**result)
        return {"chunk_type": route.chunk_type, "section": route.section}
    except Exception as e:
        return {"chunk_type": "prose", "section": "Any"}


# Test
test_queries = [
    "What was Apple total net sales in Q2 2026?",
    "Why did Apple iPhone revenue increase?",
    "What was Apple total assets?",
    "What was Apple diluted EPS Q2 2026?",
    "How much did Apple pay in dividends?",
     "What was Apple share repurchase activity in Q2 2026?"
]

print("Query Router test:\n")
for q in test_queries:
    route = route_query(q)
    print(f"Q: {q}")
    print(f"   → {route}\n")

Query Router test:

Q: What was Apple total net sales in Q2 2026?
   → {'chunk_type': 'table', 'section': 'Income Statement'}

Q: Why did Apple iPhone revenue increase?
   → {'chunk_type': 'prose', 'section': 'MD&A'}

Q: What was Apple total assets?
   → {'chunk_type': 'table', 'section': 'Balance Sheet'}

Q: What was Apple diluted EPS Q2 2026?
   → {'chunk_type': 'table', 'section': 'Notes'}

Q: How much did Apple pay in dividends?
   → {'chunk_type': 'table', 'section': 'Cash Flow'}

Q: What was Apple share repurchase activity in Q2 2026?
   → {'chunk_type': 'prose', 'section': 'Notes'}



# Retrieval

Step 12 :  Retrieval functions

In [29]:
NOISE_SECTIONS = ["General", "Signature", "Exhibits"]


def retrieve_with_parent_child(
    query: str,
    ticker: str,
    section: str = None,
    chunk_type: str = None,
    top_k: int = 5,
    alpha: float = 0.5
) -> list:

    namespace = f"{ticker}_PC"

    # Dense embedding via SentenceTransformer
    dense_vector = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Sparse embedding via BM25
    sparse_vector = bm25_encoder.encode_queries(query)

    # Build filter
    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    if section and section != "Any":
        filter_dict["section"] = {"$eq": section}
    if chunk_type:
        filter_dict["chunk_type"] = {"$eq": chunk_type}

    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k * 3,
        namespace=namespace,
        include_metadata=True,
        filter=filter_dict
    )

    seen_parents = set()
    retrieved = []

    for match in results.matches:
        parent_id = match.metadata.get("parent_id")
        if not parent_id or parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)
        parent = docstore.get(parent_id)
        if parent:
            retrieved.append({
                "child_score": match.score,
                "child_content": match.metadata.get("content"),
                "parent_content": parent["content"],
                "metadata": {**match.metadata, "parent_id": parent_id}
            })
        if len(retrieved) >= top_k:
            break

    return retrieved


def retrieve_with_router(
    query: str,
    ticker: str,
    top_k: int = 5,
    alpha: float = 0.5
) -> tuple:

    route = route_query(query)
    retrieved = retrieve_with_parent_child(
        query=query,
        ticker=ticker,
        section=route["section"],
        chunk_type=route["chunk_type"],
        top_k=top_k,
        alpha=alpha
    )
    return retrieved, route


# Load docstore
with open("../data/processed/docstore.json", "r") as f:
    docstore = json.load(f)

print(f"✓ Docstore loaded: {len(docstore)} parents")

# Quick test
query = "What was Apple total net sales in Q2 2026?"
results, route = retrieve_with_router(query, "AAPL", top_k=3)

print(f"\nQuery : {query}")
print(f"Route : {route}")
print(f"Results: {len(results)}")
for i, r in enumerate(results, 1):
    print(f"\nResult {i} | score: {r['child_score']:.4f}")
    print(f"  Section: {r['metadata'].get('section')}")
    print(f"  Child  : {r['child_content'][:520]}")

✓ Docstore loaded: 656 parents

Query : What was Apple total net sales in Q2 2026?
Route : {'chunk_type': 'table', 'section': 'Income Statement'}
Results: 2

Result 1 | score: 0.1862
  Section: Income Statement
  Child  : |                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|----------------------------------------------|----------------------|----------------------|--------------------|--------------------|
| Total net sales                              | 111,184              | 95,359               | 254,940            | 219,659            |

Result 2 | score: 0.1548
  Section: Income Statement
  Child  : |                                                                              | Three Months Ended   | Three Months Ended   | Six Months Ended   | Six Months Ended   |
|------------------------------------------------------------------------------|----------------------|----------------------|-------

In [ ]:
GOLDEN_QUERIES = [
    {"id": "Q001", "query": "What was Apple total net sales in Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q002", "query": "What was Apple net income in Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q003", "query": "What was Apple gross margin in Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q004", "query": "What was Apple research and development expense Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q005", "query": "What was Apple diluted earnings per share Q2 2026?", "ticker": "AAPL", "expected_section": "Notes", "expected_chunk_type": "table"},
    {"id": "Q006", "query": "What was Apple total assets in Q2 2026?", "ticker": "AAPL", "expected_section": "Balance Sheet", "expected_chunk_type": "table"},
    {"id": "Q007", "query": "What was Apple cash and cash equivalents Q2 2026?", "ticker": "AAPL", "expected_section": "Balance Sheet", "expected_chunk_type": "table"},
    {"id": "Q008", "query": "By what percentage did Apple total net sales grow in Q2 2026?", "ticker": "AAPL", "expected_section": "MD&A", "expected_chunk_type": "prose"},
    {"id": "Q009", "query": "Why did Apple iPhone net sales increase in Q2 2026?", "ticker": "AAPL", "expected_section": "MD&A", "expected_chunk_type": "prose"},
    {"id": "Q010", "query": "What was Apple cash generated by operating activities in H1 2026?", "ticker": "AAPL", "expected_section": "Cash Flow", "expected_chunk_type": "table"},
    {"id": "Q011", "query": "How much did Apple pay in dividends during H1 2026?", "ticker": "AAPL", "expected_section": "Cash Flow", "expected_chunk_type": "table"},
    {"id": "Q012", "query": "What was Apple share repurchase activity in Q2 2026?", "ticker": "AAPL", "expected_section": "Notes", "expected_chunk_type": "prose"},
    {"id": "Q013", "query": "What was Microsoft total revenue in Q1 FY2026?", "ticker": "MSFT", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q014", "query": "What was Microsoft net income in Q1 FY2026?", "ticker": "MSFT", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q015", "query": "What were Apple risk factors related to artificial intelligence?", "ticker": "AAPL", "expected_section": "Risk Factors", "expected_chunk_type": "prose"},
]

















def evaluate_retrieval(golden_queries: list, top_k: int = 5, alpha: float = 0.5) -> tuple:
    results = []

    for gq in golden_queries:
        matches, route = retrieve_with_router(
            query=gq["query"],
            ticker=gq["ticker"],
            top_k=top_k,
            alpha=alpha
        )

        hit = False
        reciprocal_rank = 0
        relevant_count = 0

        for rank, match in enumerate(matches, 1):
            section_match = match["metadata"].get("section") == gq["expected_section"]
            type_match = match["metadata"].get("chunk_type") == gq["expected_chunk_type"]

            if section_match and type_match:
                relevant_count += 1
                if not hit:
                    hit = True
                    reciprocal_rank = 1 / rank

        results.append({
            "id": gq["id"],
            "query": gq["query"],
            "ticker": gq["ticker"],
            "expected_section": gq["expected_section"],
            "routed_section": route["section"],
            "hit": hit,
            "reciprocal_rank": reciprocal_rank,
            "precision_at_k": relevant_count / top_k
        })

    hit_rate = sum(r["hit"] for r in results) / len(results)
    mrr = sum(r["reciprocal_rank"] for r in results) / len(results)
    avg_precision = sum(r["precision_at_k"] for r in results) / len(results)

    return results, {
        "hit_rate": round(hit_rate, 4),
        "mrr": round(mrr, 4),
        "avg_precision_at_k": round(avg_precision, 4),
        "top_k": top_k,
        "total_queries": len(results)
    }


print("Running evaluation...")
results, metrics = evaluate_retrieval(GOLDEN_QUERIES, top_k=5)

print("\n" + "="*50)
print("EVALUATION RESULTS — Groq + SentenceTransformer")
print("="*50)
print(f"Hit Rate@{metrics['top_k']}     : {metrics['hit_rate']}")
print(f"MRR              : {metrics['mrr']}")
print(f"Avg Precision@{metrics['top_k']} : {metrics['avg_precision_at_k']}")
print(f"Total queries    : {metrics['total_queries']}")

print("\nPer query breakdown:")
print("-"*70)
for r in results:
    status = "✓" if r["hit"] else "✗"
    routing = "→" if r["routed_section"] == r["expected_section"] else "✗route"
    print(f"{status} {r['id']} | {r['ticker']} | {r['expected_section']:<20} | routed={r['routed_section']:<20} {routing} | RR={r['reciprocal_rank']:.2f}")

Running evaluation...

EVALUATION RESULTS — Groq + SentenceTransformer
Hit Rate@5     : 1.0
MRR              : 1.0
Avg Precision@5 : 0.6133
Total queries    : 15

Per query breakdown:
----------------------------------------------------------------------
✓ Q001 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q002 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q003 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q004 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q005 | AAPL | Notes                | routed=Notes                → | RR=1.00
✓ Q006 | AAPL | Balance Sheet        | routed=Balance Sheet        → | RR=1.00
✓ Q007 | AAPL | Balance Sheet        | routed=Balance Sheet        → | RR=1.00
✓ Q008 | AAPL | MD&A                 | routed=MD&A                 → | RR=1.00
✓ Q009 | AAPL | MD&A                 | routed=MD&A                 → | RR=1.00
✓ Q010 | AAPL | Cash Flow         